In [184]:
from playwright.async_api import async_playwright
import pandas as pd

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)
    page = await browser.new_page()

    await page.goto(
        "https://basketball.realgm.com/ncaa/stats/2026/Averages/All/All/Season/All/",
        wait_until="networkidle",
        timeout=120_000
    )

    # Wait until the table exists AND bootstrap-table is attached
    await page.wait_for_function(
        """
        () => {
            const table = document.querySelector(
                'table.table.table-striped.table-centered.table-hover.table-bordered.table-compact.table-nowrap'
            );
            return table && window.$ && typeof $(table).bootstrapTable === 'function';
        }
        """,
        timeout=60_000
    )

    # 🔑 Pull ALL rows from bootstrap-table (not DOM)
    data = await page.evaluate(
        """
        () => {
            const table = document.querySelector(
                'table.table.table-striped.table-centered.table-hover.table-bordered.table-compact.table-nowrap'
            );
            return $(table).bootstrapTable('getData');
        }
        """
    )

    df = pd.DataFrame(data)
    print("Rows scraped:", len(df))  # ≈ 5052

    await browser.close()


Rows scraped: 5052


In [185]:
import html

df = df[[str(i) for i in range(1, 23)]]
df["player"] = (
    df["1"]
    .str.replace(r"<a[^>]*>", "", regex=True)
    .str.replace(r"</a>", "", regex=True)
    .str.strip()
)

df["team"] = (
    df["2"]
    .str.replace(r"<a[^>]*>", "", regex=True)
    .str.replace(r"</a>", "", regex=True)
    .str.strip()
)

df["player"] = df["player"].apply(lambda x: html.unescape(x) if isinstance(x, str) else x)
df["team"] = df["team"].apply(lambda x: html.unescape(x) if isinstance(x, str) else x)


In [ ]:
from sre_constants import FAILURE


ABBR_TO_FULL = {
    "AAMU": "Alabama A&M",
    "AF" : "Air Force",
    "BYU": "BYU",
    "TARST": "Tarleton State",
    "DUKE": "Duke",
    "NWU": "Northwestern",
    "KSU": "Kansas State",
    "PVAMU": "Prairie View",
    "MISST": "Mississippi State",
    "MINN": "Minnesota",
    "STAN": "Stanford",
    "U of U": "Utah",
    "UA": "Alabama",
    "ELON": "Elon",
    "BELL": "Bellarmine",
    "SUBR": "Southern",
    "UIW": "Incarnate Word",
    "HOF": "Hofstra",
    "STTOM": "St. Thomas",
    "TXTCH": "Texas Tech",
    "CALBU": "California Baptist",
    "BAYL": "Baylor",
    "UMFL": "Miami (FL)",
    "POLY": "Cal Poly",
    "ECAR": "East Carolina",
    "KU": "Kansas",
    "OSU": "Ohio State",
    "AKR": "Akron",
    "SMU": "SMU",
    "IPFW": "Purdue Fort Wayne",
    "SODAK": "South Dakota",
    "USC": "USC",
    "USU": "Utah State",
    "SJSU": "San Jose State",
    "AUB": "Auburn",
    "MVSU": "Mississippi Valley State",
    "CCSU": "Central Connecticut",
    "BUFF": "Buffalo",
    "RADF": "Radford",
    "LVILL": "Louisville",
    "ARK": "Arkansas",
    "COR": "Cornell",
    "UWISC": "Wisconsin",
    "WF": "Wake Forest",
    "UNC": "UNC",
    "PORTL": "Portland",
    "STMRY": "Saint Mary's",
    "UWG": "West Georgia",
    "KENN": "Kennesaw State",
    "SAMF": "Samford",
    "IU": "Indiana",
    "KENT": "Kent State",
    "CSUN": "Cal State Northridge",
    "CAM": "Campbell",
    "FRSNO": "Fresno State",
    "JKSN": "Jackson State",
    "LEHI": "Lehigh",
    "NCCU": "North Carolina Central",
    "UNF": "North Florida",
    "STFPA": "Stephen F. Austin",
    "TULAN": "Tulane",
    "MONT": "Montana",
    "SB": "Stony Brook",
    "VMI": "VMI",
    "IWAST": "Iowa State",
    "DART": "Dartmouth",
    "ND": "Notre Dame",
    "UGA": "Georgia",
    "UWASH": "Washington",
    "UCLA": "UCLA",
    "NC A&T": "North Carolina A&T",
    "WKU": "Western Kentucky",
    "ALAST": "Alabama State",
    "FIU": "Florida International",
    "ZAGS": "Gonzaga",
    "PENN": "Penn",
    "WICHI": "Wichita State",
    "HWRD": "Howard",
    "NOCOL": "Northern Colorado",
    "LBRTY": "Liberty",
    "IDAHO": "Idaho",
    "CHASO": "Charleston Southern",
    "PROV": "Providence",
    "LBSU": "Long Beach State",
    "IOWA": "Iowa",
    "OKST": "Oklahoma State",
    "DEN": "Denver",
    "TENN": "Tennessee",
    "GMU": "George Mason",
    "USNA": "Navy",
    "UMD": "Maryland",
    "UTSA": "UTSA",
    "UVM": "Vermont",
    "GAST": "Georgia State",
    "UNCA": "UNC Asheville",
    "SOMIS": "Southern Miss",
    "YSU": "Youngstown State",
    "UAPB": "Arkansas-Pine Bluff",
    "DRAKE": "Drake",
    "LAFAY": "Lafayette",
    "GCU": "Grand Canyon",
    "FLA": "Florida",
    "JMU": "James Madison",
    "VANDY": "Vanderbilt",
    "NKU": "Northern Kentucky",
    "OREG": "Oregon",
    "SCU": "Santa Clara",
    "SFA": "Stephen F. Austin",
    "SIUE": "SIU-Edwardsville",
    "OHIO": "Ohio",
    "CSUS": "Sacramento State",
    "QUIN": "Quinnipiac",
    "NEU": "Northeastern",
    "MIAMI": "Miami (OH)",
    "UNLV": "UNLV",
    "VILL": "Villanova",
    "PRIN": "Princeton",
    "ALBNY": "Albany (NY)",
    "OMAHA": "Omaha",
    "DUQ": "Duquesne",
    "ULM": "Louisiana-Monroe",
    "UNT": "North Texas",
    "WVU": "West Virginia",
    "DAY": "Dayton",
    "ASU": "Arizona State",
    "FORD": "Fordham",
    "TROY": "Troy",
    "USA": "South Alabama",
    "UMass": "UMass",
    "VT": "Virginia Tech",
    "YALE": "Yale",
    "WOFF": "Wofford",
    "LSU": "LSU",
    "TEXAS": "Texas",
    "UVA": "Virginia",
    "ODU": "Old Dominion",
    "MARQ": "Marquette",
    "LIU": "LIU",
    "OKLA": "Oklahoma",
    "XAV": "Xavier",
    "GT": "Georgia Tech",
    "UCI": "UC-Irvine",
    "HAR": "Harvard",
    "U of H": "Houston",
    "EIU": "Eastern Illinois",
    "OAKL": "Oakland",
    "COLO": "Colorado",
    "APSU": "Austin Peay",
    "UNCW": "UNC Wilmington",
    "UAB": "UAB",
    "ORU": "Oral Roberts",
    "UConn": "UConn",
    "UNCG": "UNC Greensboro",
    "C of C": "College of Charleston",
    "STET": "Stetson",
    "ACU": "Abilene Christian",
    "WSU": "Washington State",
    "ILL": "Illinois",
    "IONA": "Iona",
    "GTOWN": "Georgetown",
    "SEA": "Seattle",
    "FGCU": "Florida Gulf Coast",
    "CCU": "Coastal Carolina",
    "MICH": "Michigan",
    "DEL": "Delaware",
    "URI": "Rhode Island",
    "RUTG": "Rutgers",
    "UCSB": "UCSB",
    "PENST": "Penn State",
    "UMBC": "UMBC",
    "SYRA": "Syracuse",
    "MARSH": "Marshall",
    "UK": "Kentucky",
    "MISS": "Ole Miss",
    "APPST": "Appalachian State",
    "ETSU": "ETSU",
    "TULSA": "Tulsa",
    "LAMAR": "Lamar",
    "PURD": "Purdue",
    "UTA": "UT Arlington",
    "TXST": "Texas State",
    "NCST": "NC State",
    "UCF": "UCF",
    "RICE": "Rice",
    "TCU": "TCU",
    "VCU": "VCU",
    "SDSU": "San Diego State",
    "STLOU": "Saint Louis",
    "MTSU": "Middle Tennessee",
    "SETON": "Seton Hall",
    "HAW": "Hawaii",
    "NIAG": "Niagara",
    "RICH": "Richmond",
    "UND": "North Dakota",
    "UNH": "New Hampshire",
    "FAMU": "Florida A&M",
    "CREI": "Creighton",
    "NJIT": "NJIT",
    "RIDER": "Rider",
    "CLEM": "Clemson",
    "SELU": "Southeastern Louisiana",
    "TAMU-CC": "Texas A&M-Corpus Christi",
    "MAINE": "Maine",
    "JKSNV": "Jacksonville",
    "IUI" : "IU Indianapolis",
    "FAIR" : "Fairfield",
    "MER" : "Mercer",
    "TEM" : "Temple",
    "BGSU" : "Bowling Green",
    "Cal" : "California",
    "PACI" : "Pacific",
    "FRMN" : "Furman",
    "MZZST" : "Missouri State",
    "BRDLY" : "Bradley",
    "STBN" : "St. Bonaventure",
    "TNST" : "Tennessee State",
}

df[~df.team.isin(ABBR_TO_FULL.keys())].team

/var/folders/p_/d5kqctzj6579dv531381_klh0000gn/T/ipykernel_2353/3584650765.py:1: DeprecationWarning: module 'sre_constants' is deprecated
  from sre_constants import FAILURE


123     BRDLY
124      STBN
126      TNST
132       COM
139        LA
        ...  
5043      GWU
5047      CAN
5048    GRAMB
5050    TEXSO
5051     MRHD
Name: team, Length: 2111, dtype: object

In [175]:
import psycopg2
conn = psycopg2.connect(
    dbname="ncaa",
    host="localhost",
    port=5432
)

pd.read_sql("SELECT DISTINCT team FROM players_box_w_conferences WHERE conference IS NOT NULL", conn).team.unique()


/var/folders/p_/d5kqctzj6579dv531381_klh0000gn/T/ipykernel_2353/59097103.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT DISTINCT team FROM players_box_w_conferences WHERE conference IS NOT NULL", conn).team.unique()


array(['Texas-Rio Grande Valley', 'New Mexico', 'Texas State', 'Arkansas',
       'Holy Cross', 'UTSA', "Mount St. Mary's", 'Bucknell', 'Washington',
       'Campbell', 'Southern Indiana', 'Georgetown', 'Fordham',
       'New Haven', 'UMass', 'Syracuse', 'Cincinnati', 'San Diego State',
       'Youngstown State', 'Rutgers', 'Cal State Northridge', 'Merrimack',
       'Southeast Missouri State', 'Louisiana', 'Alcorn State',
       'Western Michigan', 'LIU', 'Georgia Tech', 'Gardner-Webb',
       'California Baptist', 'Richmond', 'Bellarmine',
       'Jacksonville State', 'St. Thomas', 'New Orleans', 'Mercyhurst',
       'Long Beach State', 'SIU-Edwardsville', 'Southern Illinois',
       'Ball State', 'Stetson', 'Hampton', 'TCU', 'UT Arlington',
       'Columbia', 'Fresno State', 'Creighton', 'Bethune-Cookman',
       'Boise State', 'Oral Roberts', 'Abilene Christian',
       'Houston Christian', 'Sacred Heart', 'San Francisco', 'Valparaiso',
       'Niagara', 'Ohio', 'Jacksonville', 'Th